# Financial Conditions - Notebook 4: Analysis & Reporting


## 1. Load All Outputs From Notebooks 1-3


In [ ]:
# Cell 1: Drive mount (top of every notebook)
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/macro_strategies/'
import os
os.makedirs(BASE_PATH + 'data/raw', exist_ok=True)
os.makedirs(BASE_PATH + 'data/processed', exist_ok=True)


In [ ]:
import pandas as pd
from src.reporting import (
    plot_signal, plot_cumulative_pnl, plot_drawdown,
    performance_metrics, metrics_table_markdown, save_all_figures,
)
panel = pd.read_parquet(BASE_PATH + 'data/processed/aligned_panel.parquet')
signal = pd.read_parquet(BASE_PATH + 'data/processed/signals.parquet')['signal']


## 2. Executive Summary Statistics


In [ ]:
import pickle

with open(BASE_PATH + 'data/processed/backtest_results.pkl', 'rb') as f:
    results = pickle.load(f)

returns_mkt = pd.read_parquet(BASE_PATH + 'data/processed/market_returns.parquet')
common = results['directional'].returns().index
r_bench = returns_mkt['SPY'].reindex(common)

for label, pf in [('Directional', results['directional']), ('Overlay', results['overlay'])]:
    m = performance_metrics(pf.returns(), freq=12, benchmark=r_bench)
    print(metrics_table_markdown(m, title=label))
    print()

m_bench = performance_metrics(r_bench, freq=12, benchmark=r_bench)
print(metrics_table_markdown(m_bench, title='Benchmark (SPY)'))

## 3. Signal Deep-Dive

Behaviour during named macro episodes: GFC, COVID, 2022 inflation shock.


In [ ]:
EVENTS = {
    'GFC':        '2008-09-15',
    'COVID':      '2020-03-15',
    '2022 CPI shock': '2022-06-01',
}
plot_signal(signal, title='Signal (z-score) with named events', events=EVENTS)


## 4. Cross-Strategy Correlation


In [ ]:
# Compare against other strategies' signals if available


## 5. Regime Analysis

4-quadrant (Growth x Inflation) conditional Sharpe ratios.


In [ ]:
from src.composite import investment_clock_regime

sig_df = pd.read_parquet(BASE_PATH + 'data/processed/signals.parquet')
growth_proxy = sig_df['signal'] if 'growth_signal' not in sig_df.columns else sig_df['growth_signal']
inflation_proxy = sig_df['signal']

regime = investment_clock_regime(growth_proxy, inflation_proxy)
r_dir = results['directional'].returns()
regime_aligned = regime.reindex(r_dir.index).ffill()

print("Regime distribution:")
print(regime_aligned.value_counts().to_string())
print()
print("Conditional Sharpe — Directional strategy:")
for reg in ['Recovery', 'Expansion', 'Stagflation', 'Deflation']:
    r_sl = r_dir[regime_aligned == reg]
    if len(r_sl) > 3:
        ann_r = (1 + r_sl.mean()) ** 12 - 1
        ann_v = r_sl.std() * 12 ** 0.5
        sharpe = ann_r / ann_v if ann_v > 0 else float('nan')
        print(f"  {reg:15s}: Sharpe={sharpe:.2f}  (n={len(r_sl)} months)")

## 6. Export for GitHub Pages


In [ ]:
import matplotlib.pyplot as plt

returns_mkt = pd.read_parquet(BASE_PATH + 'data/processed/market_returns.parquet')
r_bench = returns_mkt['SPY'].reindex(results['directional'].returns().index)

returns_multi = pd.DataFrame({
    'Directional': results['directional'].returns(),
    'Overlay':     results['overlay'].returns(),
    'SPY':         r_bench,
}).dropna()

EVENTS = {
    'GFC':            '2008-09-15',
    'COVID':          '2020-03-15',
    '2022 CPI shock': '2022-06-01',
}

fig_signal = plot_signal(signal, title='Signal (z-score)', events=EVENTS, save_path=None)
fig_pnl    = plot_cumulative_pnl(returns_multi, title='Cumulative PnL', save_path=None)
fig_dd     = plot_drawdown(results['directional'].returns(), title='Drawdown', save_path=None)

saved = save_all_figures(
    [('signal_timeseries', fig_signal),
     ('pnl_cumulative',    fig_pnl),
     ('drawdown',          fig_dd)],
    out_dir='assets/charts',
)
for p in saved:
    print(f'Saved: {p}')
plt.show()

## 7. Limitations & Caveats

- Signal assumes regime stationarity over the z-score window.
- Benchmark correlations shift during crises.
- Some ALFRED series are quarterly -- monthly resampling introduces artefacts.
